# Preprocesamiento y limpieza del corpus


Este notebook parte del corpus en bruto (corpus_noticias_final_RAW.csv), obtenido tras la fase de recolección manual de noticias de El País, y aplica limpieza, detección de duplicados y falsos positivos, y asignación de fases históricas, generando el corpus final de análisis (corpus_limpio_final.csv).

#### 1. Importar librerias

In [284]:
#libreria para normalizar texto sin tildes ni caracteres especiales
!pip install unidecode

In [285]:
import pandas as pd
import yaml
from unidecode import unidecode
from collections import Counter

#### 2.Cargar el CSV con el corpus 

In [286]:
df_corpus_ppal = pd.read_csv("corpus_noticias_final_RAW.csv")
df_corpus_ppal.head(5)
df_corpus_ppal.tail(5)

,fecha,hora,zona_horaria,antetitulo,titulo,autor,subtitulo,url,palabra_clave,pagina_busqueda,posicion,fecha_dt,cuerpo_noticia,longitud_cuerpo
133,19/01/2011,14:39,CET,NaN,Una oposición dividida y débil,Ignacio Cembrero,El sindicato único UGTT emerge como principal ...,https://elpais.com/diario/2011/01/19/internaci...,UGTT,1,5,2011-01-19 00:00:00,Una oposición dividida y débil El sindicato ún...,3773
134,26/01/2011,07:00,CET,NaN,"Reforma o ruptura, el dilema de Túnez",Juan Miguel Muñoz,La parálisis por la falta de gobierno lleva al...,https://elpais.com/diario/2011/01/26/internaci...,UGTT,3,3,2011-01-26 00:00:00,"Reforma o ruptura, el dilema de Túnez La parál...",3827
135,22/01/2011,07:00,CET,NaN,Comunistas e islamistas mantienen la protesta ...,Juan Miguel Muñoz,El Ejecutivo decide reabrir las escuelas y rea...,https://elpais.com/diario/2011/01/22/internaci...,UGTT,3,19,2011-01-22 00:00:00,Comunistas e islamistas mantienen la protesta ...,4030
136,19/01/2011,07:00,CET,NaN,El Gobierno de Túnez se resquebraja,Juan Miguel Muñoz,Cuatro ministros dimiten del nuevo Ejecutivo p...,https://elpais.com/diario/2011/01/19/internaci...,UGTT,3,20,2011-01-19 00:00:00,El Gobierno de Túnez se resquebraja Cuatro min...,5734
137,28/01/2011,16:54,CET,NaN,La «calle» tunecina,NaN,NaN,https://elpais.com/internacional/2011/01/28/ac...,UGTT,5,15,2011-01-28 00:00:00,"La «calle» tunecina En Túnez, sigue el pulso e...",4816


#### 3. Exploración inicial

In [287]:
df_corpus_ppal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   fecha            138 non-null    object 
 1   hora             138 non-null    object 
 2   zona_horaria     138 non-null    object 
 3   antetitulo       0 non-null      float64
 4   titulo           138 non-null    object 
 5   autor            84 non-null     object 
 6   subtitulo        112 non-null    object 
 7   url              138 non-null    object 
 8   palabra_clave    138 non-null    object 
 9   pagina_busqueda  138 non-null    int64  
 10  posicion         138 non-null    int64  
 11  fecha_dt         138 non-null    object 
 12  cuerpo_noticia   138 non-null    object 
 13  longitud_cuerpo  138 non-null    int64  
dtypes: float64(1), int64(3), object(10)
memory usage: 15.2+ KB


#### 4. Crear copia del corpus original para hacer ahí las transformaciones

In [288]:
df_corpus_limpio = df_corpus_ppal.copy()

#### 5. Formato fecha


Eliminar columna de fecha_dt auxiliar

In [289]:
df_corpus_limpio = df_corpus_limpio.drop(columns=["fecha_dt"])

Adaptar la fecha inicial date en el formato correspondiente para poder usarla a posteriori en las visualizaciones

In [290]:
df_corpus_limpio["fecha"] = pd.to_datetime(df_corpus_limpio["fecha"],dayfirst=True,errors="coerce")


Adaptar campo hora a formato hora de datetime

In [291]:
df_corpus_limpio["hora"] = pd.to_datetime(df_corpus_limpio["hora"],format="%H:%M",errors="coerce").dt.time

In [292]:
df_corpus_limpio.head(2)


,fecha,hora,zona_horaria,antetitulo,titulo,autor,subtitulo,url,palabra_clave,pagina_busqueda,posicion,cuerpo_noticia,longitud_cuerpo
0,2011-01-23,07:02:00,CET,NaN,La llama que incendió Túnez,Juan Miguel Muñoz,La inmolación del vendedor de frutas Mohamed B...,https://elpais.com/diario/2011/01/23/domingo/1...,Mohamed Bouazizi,3,4,La llama que incendió Túnez La inmolación del ...,21592
1,2011-01-17,15:23:00,CET,NaN,Un egipcio y un mauritano se queman a lo bonzo,NaN,Se suman un argelino y a otros dos tunecinos q...,https://elpais.com/internacional/2011/01/17/ac...,Mohamed Bouazizi,3,10,Un egipcio y un mauritano se queman a lo bonzo...,5781


Verificación de que la hora está en formato datetime

In [293]:
type(df_corpus_limpio.loc[0, "hora"])

datetime.time

## Análisis Exploratorio de Datos (EDA)

#### 4. Detección de falsos positivos

In [294]:

with open("tunez.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

terminos = []
for grupo in ["place", "people", "organizations", "concepts", "narrative_markers"]:
    for item in config.get(grupo, []):
        term = item["term"]
        for variante in [term] + item.get("variants", []):
            terminos.append((grupo, term, unidecode(variante).lower()))

df_corpus_limpio['texto_completo'] = (
    df_corpus_limpio['titulo'].fillna('') + ' ' + df_corpus_limpio['cuerpo_noticia'].fillna('')
).map(lambda x: unidecode(str(x)).lower())

def detectar(texto):
    encontrados = {}
    for grupo, term_canon, variante_norm in terminos:
        if variante_norm in texto:
            encontrados[term_canon] = grupo
    return encontrados

resultados = df_corpus_limpio['texto_completo'].map(detectar)
df_corpus_limpio['entidades_detectadas'] = resultados.map(lambda d: sorted(set(d.keys())))
df_corpus_limpio['n_entidades'] = df_corpus_limpio['entidades_detectadas'].map(len)

Expicación metodológica:

La columna palabra_clave refleja el término introducido manualmente en el momento de la recogida y puede contener inexactitudes puntuales debido a la naturaleza manual del proceso de búsqueda. Por este motivo, se ha creado el campo de entidades_detectadas y el análisis de contenido se basará en este campo, calculada de forma automática y reproducible sobre el texto completo de cada artículo, en lugar de en la keyword de búsqueda registrada.

In [295]:
candidatos_falsos_positivos = df_corpus_limpio[df_corpus_limpio['n_entidades'] == 0]
print(f"Candidatos encontrados: {len(candidatos_falsos_positivos)}")
print(candidatos_falsos_positivos[['fecha', 'titulo', 'palabra_clave']].to_string())

Candidatos encontrados: 3
        fecha                                                                                            titulo palabra_clave
74 2010-08-21                                                      Bouza estrena el medallero español en Poznan   Sidi Bouzid
76 2011-01-13  Sidi Hoteles cerrará sus establecimientos de El Saler y San Juan y despedirá a sus 170 empleados   Sidi Bouzid
79 2011-08-31                                      El BNG opta por la continuidad en los candidatos al Congreso   Sidi Bouzid


Expicación metodológica: 

Se han identificado 3 artículos que no contienen ninguna entidad relevante del esquema de configuración (tunez.yaml) en el título ni en el cuerpo del texto. Tras revisión manual, se confirma que corresponden a contenido ajeno al objeto de estudio.

In [296]:
antes = len(df_corpus_limpio)

candidatos_falsos_positivos = df_corpus_limpio[df_corpus_limpio['n_entidades'] == 0]
print("Se eliminan los siguientes artículos (confirmados como ruido):")
print(candidatos_falsos_positivos[['fecha', 'titulo', 'palabra_clave']].to_string())

df_corpus_limpio = df_corpus_limpio[df_corpus_limpio['n_entidades'] > 0].copy()

print(f"\nEliminados: {antes - len(df_corpus_limpio)} artículos")
print(f"Filas restantes: {len(df_corpus_limpio)}")

Se eliminan los siguientes artículos (confirmados como ruido):
        fecha                                                                                            titulo palabra_clave
74 2010-08-21                                                      Bouza estrena el medallero español en Poznan   Sidi Bouzid
76 2011-01-13  Sidi Hoteles cerrará sus establecimientos de El Saler y San Juan y despedirá a sus 170 empleados   Sidi Bouzid
79 2011-08-31                                      El BNG opta por la continuidad en los candidatos al Congreso   Sidi Bouzid

Eliminados: 3 artículos
Filas restantes: 135


In [297]:
# Se elimina la columna auxiliar que unía título con cuerpo_noticia y no normalizaba para encontrar los términos en el YALM
df_corpus_limpio = df_corpus_limpio.drop(columns=['texto_completo'])

In [298]:
df_corpus_limpio.head(2)

,fecha,hora,zona_horaria,antetitulo,titulo,autor,subtitulo,url,palabra_clave,pagina_busqueda,posicion,cuerpo_noticia,longitud_cuerpo,entidades_detectadas,n_entidades
0,2011-01-23,07:02:00,CET,NaN,La llama que incendió Túnez,Juan Miguel Muñoz,La inmolación del vendedor de frutas Mohamed B...,https://elpais.com/diario/2011/01/23/domingo/1...,Mohamed Bouazizi,3,4,La llama que incendió Túnez La inmolación del ...,21592,"[Ben Ali, Mohamed Bouazizi, RCD, Rachid Ammar,...",8
1,2011-01-17,15:23:00,CET,NaN,Un egipcio y un mauritano se queman a lo bonzo,NaN,Se suman un argelino y a otros dos tunecinos q...,https://elpais.com/internacional/2011/01/17/ac...,Mohamed Bouazizi,3,10,Un egipcio y un mauritano se queman a lo bonzo...,5781,"[Ben Ali, Mohamed Bouazizi, Túnez]",3


#### 5. Detección de duplicados

Una misma noticia puede aparecer en dos URLs distintas (versión web y edición impresa del día siguiente).
Se comparan los textos para detectar y eliminar estos duplicados del corpus.

In [299]:
df_corpus_limpio['titulo'].duplicated().sum()

np.int64(11)

In [300]:
df_corpus_limpio[df_corpus_limpio['titulo'].duplicated(keep=False)][['titulo', 'url', 'subtitulo', 'cuerpo_noticia']].sort_values(["titulo"])

,titulo,url,subtitulo,cuerpo_noticia
20,"""Nos han robado la revolución""",https://elpais.com/internacional/2011/12/17/ac...,La ciudad de Túnez donde arrancó la ‘primavera...,"""Nos han robado la revolución"" La ciudad de Tú..."
24,"""Nos han robado la revolución""",https://elpais.com/diario/2011/12/18/internaci...,La ciudad de Túnez donde arrancó la 'primavera...,"""Nos han robado la revolución"" La ciudad de Tú..."
93,"Ben Ali, expresidente de Túnez, en coma en un ...",https://elpais.com/internacional/2011/02/17/ac...,El periodista Nicholas Beau precisa que el est...,"Ben Ali, expresidente de Túnez, en coma en un ..."
87,"Ben Ali, expresidente de Túnez, en coma en un ...",https://elpais.com/diario/2011/02/18/internaci...,"El dictador, de 74 años, sufrió el martes un d...","Ben Ali, expresidente de Túnez, en coma en un ..."
28,El año del jazmín,https://elpais.com/internacional/2011/12/15/de...,NaN,El año del jazmín Mohamed Bouazizi debía andar...
29,El año del jazmín,https://elpais.com/internacional/2011/12/14/ac...,El incendio que empezó en Túnez hace 12 meses ...,El año del jazmín El incendio que empezó en Tú...
43,El año del jazmín,https://elpais.com/diario/2011/12/15/internaci...,NaN,El año del jazmín Mohamed Bouazizi debía andar...
104,El policía que llegó a caudillo,https://elpais.com/internacional/2011/01/14/ac...,Ben Ali hizo carrera en los cuerpos de segurid...,El policía que llegó a caudillo Ben Ali hizo c...
77,El policía que llegó a caudillo,https://elpais.com/diario/2011/01/15/internaci...,Ben Ali hizo carrera en los cuerpos de segurid...,El policía que llegó a caudillo Ben Ali hizo c...
120,Islam y democracia,https://elpais.com/diario/2011/11/01/opinion/1...,NaN,Islam y democracia El humorista del diario tun...


- Función para normalizar el título y el inicio del texto de cada noticia, crear una clave única combinando ambos y utilizar esa clave para localizar noticias duplicadas, incluso cuando tienen URLs diferentes o pequeñas diferencias de formato.

In [301]:
def normalizar_titulo(t):
    return unidecode(str(t)).lower().strip()

def inicio_cuerpo(c, n_caracteres=80):
    return unidecode(str(c)).lower().strip()[:n_caracteres]

df_corpus_limpio['titulo_normalizado'] = df_corpus_limpio['titulo'].map(normalizar_titulo)
df_corpus_limpio['inicio_cuerpo'] = df_corpus_limpio['cuerpo_noticia'].map(inicio_cuerpo)

# clave de duplicado real: mismo título Y mismo arranque del cuerpo
df_corpus_limpio['clave_duplicado'] = df_corpus_limpio['titulo_normalizado'] + " || " + df_corpus_limpio['inicio_cuerpo']

duplicados = df_corpus_limpio[df_corpus_limpio.duplicated(subset='clave_duplicado', keep=False)]
print(f"Filas implicadas en duplicados REALES: {len(duplicados)}")
print(duplicados[['titulo', 'url']].sort_values('titulo').to_string())

Filas implicadas en duplicados REALES: 14
                                                                          titulo                                                                                         url
20                                                "Nos han robado la revolución"               https://elpais.com/internacional/2011/12/17/actualidad/1324141881_437499.html
24                                                "Nos han robado la revolución"                   https://elpais.com/diario/2011/12/18/internacional/1324162811_850215.html
28                                                             El año del jazmín  https://elpais.com/internacional/2011/12/15/del_alfiler_al_elefante/1323943200_132394.html
43                                                             El año del jazmín                   https://elpais.com/diario/2011/12/15/internacional/1323903605_850215.html
77                                               El policía que llegó a caudillo             

Eliminar los duplicados conservando la primera aparición

In [302]:

df_corpus_limpio = df_corpus_limpio.drop_duplicates(subset="clave_duplicado",keep="first")
print(f"Número de noticias tras eliminar duplicados: {len(df_corpus_limpio)}")

Número de noticias tras eliminar duplicados: 128


In [303]:
df_corpus_limpio[df_corpus_limpio['titulo'].duplicated(keep=False)][['titulo', 'url', 'subtitulo', 'cuerpo_noticia']].sort_values(["titulo"])

,titulo,url,subtitulo,cuerpo_noticia
87,"Ben Ali, expresidente de Túnez, en coma en un ...",https://elpais.com/diario/2011/02/18/internaci...,"El dictador, de 74 años, sufrió el martes un d...","Ben Ali, expresidente de Túnez, en coma en un ..."
93,"Ben Ali, expresidente de Túnez, en coma en un ...",https://elpais.com/internacional/2011/02/17/ac...,El periodista Nicholas Beau precisa que el est...,"Ben Ali, expresidente de Túnez, en coma en un ..."
28,El año del jazmín,https://elpais.com/internacional/2011/12/15/de...,NaN,El año del jazmín Mohamed Bouazizi debía andar...
29,El año del jazmín,https://elpais.com/internacional/2011/12/14/ac...,El incendio que empezó en Túnez hace 12 meses ...,El año del jazmín El incendio que empezó en Tú...
120,Islam y democracia,https://elpais.com/diario/2011/11/01/opinion/1...,NaN,Islam y democracia El humorista del diario tun...
121,Islam y democracia,https://elpais.com/internacional/2011/10/31/ac...,El nuevo régimen de Túnez puede ofrecer un eje...,Islam y democracia El nuevo régimen de Túnez p...
34,Los ciberataques colapsan todas las webs del r...,https://elpais.com/internacional/2011/01/05/ac...,Las protestas contra la dictadura provocan la ...,Los ciberataques colapsan todas las webs del r...
40,Los ciberataques colapsan todas las webs del r...,https://elpais.com/diario/2011/01/06/internaci...,Los internautas respaldan a la oposición en la...,Los ciberataques colapsan todas las webs del r...


Eliminar las filas de las noticias que tienen diferente subtitulo pero mismo cuerpo

In [304]:
df_corpus_limpio.loc[[87, 93, 34, 40], ["fecha", "titulo", "cuerpo_noticia", "subtitulo"]]

,fecha,titulo,cuerpo_noticia,subtitulo
87,2011-02-18,"Ben Ali, expresidente de Túnez, en coma en un ...","Ben Ali, expresidente de Túnez, en coma en un ...","El dictador, de 74 años, sufrió el martes un d..."
93,2011-02-17,"Ben Ali, expresidente de Túnez, en coma en un ...","Ben Ali, expresidente de Túnez, en coma en un ...",El periodista Nicholas Beau precisa que el est...
34,2011-01-05,Los ciberataques colapsan todas las webs del r...,Los ciberataques colapsan todas las webs del r...,Las protestas contra la dictadura provocan la ...
40,2011-01-06,Los ciberataques colapsan todas las webs del r...,Los ciberataques colapsan todas las webs del r...,Los internautas respaldan a la oposición en la...


In [305]:
df_corpus_limpio = df_corpus_limpio.drop(index=[87, 40])

Eliminar columnas auxiliares usadas para la detección de duplicados

In [306]:
df_corpus_limpio = df_corpus_limpio.drop(columns=["titulo_normalizado", "inicio_cuerpo", "clave_duplicado"])

Eliminar columnas sin valor analítico para este proyecto

In [307]:

df_corpus_limpio = df_corpus_limpio.drop(columns=['antetitulo', 'pagina_busqueda', 'posicion'])

In [308]:
df_corpus_limpio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 126 entries, 0 to 137
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   fecha                 126 non-null    datetime64[ns]
 1   hora                  126 non-null    object        
 2   zona_horaria          126 non-null    object        
 3   titulo                126 non-null    object        
 4   autor                 75 non-null     object        
 5   subtitulo             101 non-null    object        
 6   url                   126 non-null    object        
 7   palabra_clave         126 non-null    object        
 8   cuerpo_noticia        126 non-null    object        
 9   longitud_cuerpo       126 non-null    int64         
 10  entidades_detectadas  126 non-null    object        
 11  n_entidades           126 non-null    int64         
dtypes: datetime64[ns](1), int64(2), object(9)
memory usage: 12.8+ KB


In [309]:
df_corpus_limpio.sort_values(["fecha"])

,fecha,hora,zona_horaria,titulo,autor,subtitulo,url,palabra_clave,cuerpo_noticia,longitud_cuerpo,entidades_detectadas,n_entidades
27,2011-01-05,07:00:00,CET,Crece la protesta social en Túnez por la crisi...,NaN,NaN,https://elpais.com/diario/2011/01/05/internaci...,Sidi Bouzid,Crece la protesta social en Túnez por la crisi...,792,"[Ben Ali, Sidi Bouzid, Túnez]",3
34,2011-01-05,20:33:00,CET,Los ciberataques colapsan todas las webs del r...,NaN,Las protestas contra la dictadura provocan la ...,https://elpais.com/internacional/2011/01/05/ac...,Sidi Bouzid,Los ciberataques colapsan todas las webs del r...,5193,"[Ben Ali, Mohamed Bouazizi, Sidi Bouzid, Túnez...",5
18,2011-01-08,03:44:00,CET,Tercer muerto a lo bonzo en Túnez,EFE,NaN,https://elpais.com/diario/2011/01/08/internaci...,elecciones 2011,Tercer muerto a lo bonzo en Túnez Un joven tun...,1728,"[Mohamed Bouazizi, Sidi Bouzid, Túnez]",3
55,2011-01-10,07:00:00,CET,Las protestas sacuden al régimen de Túnez y su...,Ignacio Cembrero,El presidente Ben Alí saca al Ejército a la ca...,https://elpais.com/diario/2011/01/10/internaci...,Sidi Bouzid,Las protestas sacuden al régimen de Túnez y su...,6990,"[Ben Ali, Kasserine, Mohamed Bouazizi, Sidi Bo...",6
26,2011-01-11,00:20:00,CET,Túnez cierra todas las escuelas y las universi...,NaN,La oposición parlamentaria asegura que la poli...,https://elpais.com/internacional/2011/01/11/ac...,Sidi Bouzid,Túnez cierra todas las escuelas y las universi...,4125,"[Ben Ali, Kasserine, Sfax, Sidi Bouzid, Túnez]",5
...,...,...,...,...,...,...,...,...,...,...,...,...
14,2011-12-18,21:09:00,CET,Honda división en Túnez entre laicos\ny “barbu...,Ignacio Cembrero,Los islamistas ostentan el grueso del poder en...,https://elpais.com/internacional/2011/12/18/ac...,elecciones 2011,Honda división en Túnez entre laicos y “barbud...,7589,"[Asamblea Constituyente, Ben Ali, Ennahda, Ham...",5
117,2011-12-19,07:00:00,CET,Honda división en Túnez entre laicos y 'barbud...,Ignacio Cembrero,Los islamistas ostentan el grueso del poder en...,https://elpais.com/diario/2011/12/19/internaci...,Ennahda,Honda división en Túnez entre laicos y 'barbud...,5178,"[Asamblea Constituyente, Ben Ali, Ennahda, Ham...",5
13,2011-12-22,17:21:00,CET,Los islamistas dominan el primer Gobierno demo...,Ignacio Cembrero,"El partido vencedor Ennahda ocupa 18 carteras,...",https://elpais.com/internacional/2011/12/22/ac...,elecciones 2011,Los islamistas dominan el primer Gobierno demo...,2705,"[Asamblea Constituyente, Ennahda, Hamadi Jebal...",5
63,2011-12-27,07:00:00,CET,Gracias a los acampados,Jordi Vaquer,NaN,https://elpais.com/diario/2011/12/27/internaci...,Sidi Bouzid,Gracias a los acampados Este 2011 ha sido extr...,4888,"[Ben Ali, Mohamed Bouazizi, Sidi Bouzid]",3


#### 6. Revisión calidad del corpus

6.1 Detectar valores vacíos:

In [310]:
df_corpus_limpio.isna().sum().sort_values(ascending=False)

autor                   51
subtitulo               25
fecha                    0
hora                     0
zona_horaria             0
titulo                   0
url                      0
palabra_clave            0
cuerpo_noticia           0
longitud_cuerpo          0
entidades_detectadas     0
n_entidades              0
dtype: int64

6.2 Noticias sin cuerpo

In [311]:
sin_cuerpo = df_corpus_limpio[df_corpus_limpio["cuerpo_noticia"].isna()]
print("Noticias sin cuerpo:", len(sin_cuerpo))

Noticias sin cuerpo: 0


6.3 Longitud del cuerpo de las noticias

In [312]:
df_corpus_limpio["longitud_cuerpo"].describe()

count      126.000000
mean      4418.460317
std       2803.402677
min        302.000000
25%       2881.750000
50%       4020.000000
75%       5202.750000
max      21592.000000
Name: longitud_cuerpo, dtype: float64

In [313]:
columnas_revision = ["fecha","titulo","longitud_cuerpo","cuerpo_noticia"]
df_corpus_limpio.sort_values("longitud_cuerpo")[columnas_revision].head(10)

,fecha,titulo,longitud_cuerpo,cuerpo_noticia
6,2011-10-24,Túnez estrena las elecciones de la 'primavera ...,302,Túnez estrena las elecciones de la 'primavera ...
84,2011-06-21,Túnez condena en rebeldía al exdictador Ben Al...,462,Túnez condena en rebeldía al exdictador Ben Al...
113,2011-03-08,MIL TUNECINOS LLEGAN A SICILIA EN UNA NOCHE,474,Fotonoticia:Ola de cambio en el mundo árabe | ...
109,2011-01-13,Túnez impone el toque de queda para frenar las...,636,Túnez impone el toque de queda para frenar las...
86,2011-01-12,La revuelta de Túnez irrumpe en la capital,640,La revuelta de Túnez irrumpe en la capital Ben...
27,2011-01-05,Crece la protesta social en Túnez por la crisi...,792,Crece la protesta social en Túnez por la crisi...
122,2011-02-07,El islam y la democracia,899,El islam y la democracia Ante quienes recurren...
85,2011-01-14,Túnez pierde el miedo a Ben Ali,916,Túnez pierde el miedo a Ben Ali El mandatario ...
112,2011-01-17,Soldados y agentes del dictador huido se enfre...,961,Soldados y agentes del dictador huido se enfre...
17,2011-01-15,El dictador Ben Ali huye de Túnez acosado por ...,1099,El dictador Ben Ali huye de Túnez acosado por ...


6.4 Títulos vacíos

In [314]:
df_corpus_limpio[ df_corpus_limpio["titulo"].isna()]

,fecha,hora,zona_horaria,titulo,autor,subtitulo,url,palabra_clave,cuerpo_noticia,longitud_cuerpo,entidades_detectadas,n_entidades


6.5 Fechas perdidas

In [315]:
df_corpus_limpio[df_corpus_limpio["fecha"].isna()]

,fecha,hora,zona_horaria,titulo,autor,subtitulo,url,palabra_clave,cuerpo_noticia,longitud_cuerpo,entidades_detectadas,n_entidades


6.6 URLs duplicadas

In [316]:
duplicadas = df_corpus_limpio[df_corpus_limpio.duplicated(subset="url",keep=False)]
duplicadas

,fecha,hora,zona_horaria,titulo,autor,subtitulo,url,palabra_clave,cuerpo_noticia,longitud_cuerpo,entidades_detectadas,n_entidades


6.7 Distribución temporal

In [317]:
(df_corpus_limpio.groupby(df_corpus_limpio["fecha"].dt.to_period("M")).size())

fecha
2011-01    68
2011-02    13
2011-03     6
2011-04     3
2011-05     2
2011-06     7
2011-07     1
2011-08     2
2011-09     1
2011-10     9
2011-11     4
2011-12    10
Freq: M, dtype: int64

In [318]:
print("Fecha mínima:", df_corpus_limpio["fecha"].min())
print("Fecha máxima:", df_corpus_limpio["fecha"].max())

Fecha mínima: 2011-01-05 00:00:00
Fecha máxima: 2011-12-31 00:00:00


#### 7. Asignación de rangos temporales. División en periodos históricos

In [319]:
def fase(f):
    if f < pd.Timestamp("2010-12-17"):
        return "FUERA_DE_RANGO"  # Antes de la autoinmolación de Mohamed Bouazizi
    if f <= pd.Timestamp("2011-01-13"):
        return "Fase de protestas"  # Desde la autoinmolación (17-dic) hasta la víspera de la huida de Ben Ali
    if f <= pd.Timestamp("2011-02-28"):
        return "Caída del régimen"  # Desde la huida de Ben Ali (14-ene) hasta la dimisión de Ghannouchi (27-feb)
    if f <= pd.Timestamp("2011-10-22"):
        return "Transición democrática"  # Desde la dimisión de Ghannouchi (28-feb) hasta la víspera de las elecciones
    if f <= pd.Timestamp("2011-12-31"):
        return "Periodo post-electoral"  # Desde las elecciones (23-oct) hasta el cierre del año / formación del gobierno
    return "FUERA_DE_RANGO"  # Cualquier fecha posterior al cierre del periodo de estudio

df_corpus_limpio['periodo_fase'] = df_corpus_limpio['fecha'].map(fase)
print(df_corpus_limpio['periodo_fase'].value_counts())

periodo_fase
Caída del régimen         68
Transición democrática    24
Periodo post-electoral    21
Fase de protestas         13
Name: count, dtype: int64


In [320]:
df_corpus_limpio.head(2)

,fecha,hora,zona_horaria,titulo,autor,subtitulo,url,palabra_clave,cuerpo_noticia,longitud_cuerpo,entidades_detectadas,n_entidades,periodo_fase
0,2011-01-23,07:02:00,CET,La llama que incendió Túnez,Juan Miguel Muñoz,La inmolación del vendedor de frutas Mohamed B...,https://elpais.com/diario/2011/01/23/domingo/1...,Mohamed Bouazizi,La llama que incendió Túnez La inmolación del ...,21592,"[Ben Ali, Mohamed Bouazizi, RCD, Rachid Ammar,...",8,Caída del régimen
1,2011-01-17,15:23:00,CET,Un egipcio y un mauritano se queman a lo bonzo,NaN,Se suman un argelino y a otros dos tunecinos q...,https://elpais.com/internacional/2011/01/17/ac...,Mohamed Bouazizi,Un egipcio y un mauritano se queman a lo bonzo...,5781,"[Ben Ali, Mohamed Bouazizi, Túnez]",3,Caída del régimen


 8. Añadir un índice al corpus

In [321]:
# Reiniciar el índice por si hubiera habido filtros previos
df_corpus_limpio = df_corpus_limpio.reset_index(drop=True)

# Añadir un identificador único de artículo como primera columna
df_corpus_limpio.insert(0,"id_articulo",range(1, len(df_corpus_limpio) + 1))

In [322]:
df_corpus_limpio

,id_articulo,fecha,hora,zona_horaria,titulo,autor,subtitulo,url,palabra_clave,cuerpo_noticia,longitud_cuerpo,entidades_detectadas,n_entidades,periodo_fase
0,1,2011-01-23,07:02:00,CET,La llama que incendió Túnez,Juan Miguel Muñoz,La inmolación del vendedor de frutas Mohamed B...,https://elpais.com/diario/2011/01/23/domingo/1...,Mohamed Bouazizi,La llama que incendió Túnez La inmolación del ...,21592,"[Ben Ali, Mohamed Bouazizi, RCD, Rachid Ammar,...",8,Caída del régimen
1,2,2011-01-17,15:23:00,CET,Un egipcio y un mauritano se queman a lo bonzo,NaN,Se suman un argelino y a otros dos tunecinos q...,https://elpais.com/internacional/2011/01/17/ac...,Mohamed Bouazizi,Un egipcio y un mauritano se queman a lo bonzo...,5781,"[Ben Ali, Mohamed Bouazizi, Túnez]",3,Caída del régimen
2,3,2011-01-15,07:00:00,CET,El joven mártir que cambió el destino de un país,Gloria Rodríguez-Pina,El suicidio público de un parado prendió la ch...,https://elpais.com/diario/2011/01/15/internaci...,Mohamed Bouazizi,El joven mártir que cambió el destino de un pa...,2744,"[Ben Ali, Mohamed Bouazizi, Sidi Bouzid, Túnez]",4,Caída del régimen
3,4,2011-06-13,07:00:00,CEST,El viento de la 'primavera árabe',Tahar Ben Jelloun,Las rebeliones en curso están dirigidas contra...,https://elpais.com/diario/2011/06/13/opinion/1...,Primavera árabe,El viento de la 'primavera árabe' Las rebelion...,10077,"[Ben Ali, Túnez, exilio, primavera árabe]",4,Transición democrática
4,5,2011-10-11,07:00:00,CEST,La 'primavera árabe' pasó,Jordi Vaquer,NaN,https://elpais.com/diario/2011/10/11/internaci...,Primavera árabe,La 'primavera árabe' pasó La primavera árabe a...,4585,"[Ben Ali, Túnez, primavera árabe]",3,Transición democrática
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,122,2011-01-19,14:39:00,CET,Una oposición dividida y débil,Ignacio Cembrero,El sindicato único UGTT emerge como principal ...,https://elpais.com/diario/2011/01/19/internaci...,UGTT,Una oposición dividida y débil El sindicato ún...,3773,"[Ben Ali, RCD, Túnez, UGTT, exilio]",5,Caída del régimen
122,123,2011-01-26,07:00:00,CET,"Reforma o ruptura, el dilema de Túnez",Juan Miguel Muñoz,La parálisis por la falta de gobierno lleva al...,https://elpais.com/diario/2011/01/26/internaci...,UGTT,"Reforma o ruptura, el dilema de Túnez La parál...",3827,"[Ben Ali, Sfax, Túnez, UGTT, exilio]",5,Caída del régimen
123,124,2011-01-22,07:00:00,CET,Comunistas e islamistas mantienen la protesta ...,Juan Miguel Muñoz,El Ejecutivo decide reabrir las escuelas y rea...,https://elpais.com/diario/2011/01/22/internaci...,UGTT,Comunistas e islamistas mantienen la protesta ...,4030,"[Ben Ali, RCD, Túnez, UGTT]",4,Caída del régimen
124,125,2011-01-19,07:00:00,CET,El Gobierno de Túnez se resquebraja,Juan Miguel Muñoz,Cuatro ministros dimiten del nuevo Ejecutivo p...,https://elpais.com/diario/2011/01/19/internaci...,UGTT,El Gobierno de Túnez se resquebraja Cuatro min...,5734,"[Ben Ali, RCD, Túnez, UGTT, exilio]",5,Caída del régimen


#### 8. Guardado del corpus limpio en un CSV

In [323]:

df_corpus_limpio.to_csv("corpus_limpio_final.csv", index=False)
print(f"Guardado: corpus_limpio_final.csv — {len(df_corpus_limpio)} artículos, {df_corpus_limpio.shape[1]} columnas")

Guardado: corpus_limpio_final.csv — 126 artículos, 14 columnas
